# CartoPalette v4 — CVAE Training

**Key changes vs v3:**
1. **80% basemap-dependent scoring** — training data generated with 6 kartographic metrics
2. **Only sequential + diverging** — qualitative dropped entirely
3. **Basemap-adaptive candidates** — 4-stage pipeline (90% basemap-specific)
4. **Absolute thresholds** — no min-max normalization (scores are cross-basemap comparable)
5. **Diversity-aware labeling** — top-k palettes are both high-scoring AND visually distinct

Architecture is identical to v3 (EfficientNet-B0 + Projection Head + CVAE).
The improvement comes entirely from **better training data**.

## 0. Setup & GPU Check

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
import pandas as pd
import json
import os
import time
import math
from pathlib import Path
from PIL import Image
from collections import defaultdict
from torch.amp import autocast, GradScaler

# GPU check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.mem_get_info(0)[1] / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime → Change runtime type → A100 GPU')

print(f'PyTorch: {torch.__version__}')
print(f'Device: {device}')

## 1. Mount Google Drive & Extract Dataset

In [ ]:
import sys
import os

from google.colab import drive
drive.mount('/content/drive')

# Path to your uploaded zip on Google Drive
DRIVE_ZIP = '/content/drive/MyDrive/CartoPalette/cartopalette_dataset_v4.zip'
DATA_DIR = '/content/cartopalette_data'

# Extract (only if not already done)
if not os.path.exists(DATA_DIR):
    import zipfile
    print('Extracting dataset...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall(DATA_DIR)
    print('Done!')

# Add to Python path so we can import research.* modules for evaluation
sys.path.insert(0, DATA_DIR)

# Verify — zip contains data/processed/... structure
basemaps_dir = os.path.join(DATA_DIR, 'data', 'processed', 'basemaps')
labels_path = os.path.join(DATA_DIR, 'data', 'processed', 'labels', 'labels.csv')
splits_dir = os.path.join(DATA_DIR, 'data', 'processed', 'splits')
basemap_colors_dir = os.path.join(DATA_DIR, 'data', 'interim', 'basemap_colors')

n_basemaps = len([f for f in os.listdir(basemaps_dir) if f.endswith('.png')])
print(f'Basemaps: {n_basemaps}')

labels_df_check = pd.read_csv(labels_path)
print(f'Labels: {len(labels_df_check)}')
print(f'Columns: {list(labels_df_check.columns)}')
print(f'Scheme types: {labels_df_check["scheme_type"].unique()}')
print(f'Positive labels: {len(labels_df_check[labels_df_check["label_type"] == "positive"])}')
print(f'Splits: {labels_df_check["split"].value_counts().to_dict()}')
del labels_df_check

# Verify basemap colors available for evaluation
if os.path.exists(basemap_colors_dir):
    n_color_files = len([f for f in os.listdir(basemap_colors_dir) if f.endswith('.json')])
    print(f'Basemap color files: {n_color_files}')
else:
    print('WARNING: basemap_colors_dir not found — metric evaluation will not work')

# Verify research modules importable
try:
    from research.data_pipeline.score_palettes import compute_composite_score
    print('Scoring module imported successfully')
except ImportError as e:
    print(f'WARNING: Could not import scoring module: {e}')


## 2. Configuration

In [ ]:
# ── Hyperparameters v4 ──
# Architecture is IDENTICAL to v3 — the gains come from better training data
CONFIG = {
    # Architecture (same as v3)
    'latent_dim': 64,
    'cnn_feature_dim': 1280,
    'cnn_projection_dim': 256,
    'condition_dim': 256 + 10,       # projected CNN (256) + metadata (10) = 266
    'hidden_dim': 512,
    'max_palette_size': 9,
    'color_channels': 3,
    'palette_dim': 9 * 3,

    # Training (tuned for v4 data)
    'batch_size': 128,
    'epochs': 80,                    # v4: more epochs (v3 was 60) — data is higher quality
    'lr_cvae': 1e-4,                 # LR for projection head + CVAE
    'lr_cnn': 1e-5,                  # LR for CNN blocks 3-8 (10x lower)
    'lr_min': 1e-6,
    'weight_decay': 1e-5,
    'recon_patience': 25,            # epochs without recon improvement
    'gen_patience': 4,               # gen_val checks without composite improvement (4×5=20 epochs)

    # KL Annealing: Cyclical schedule
    'kl_weight_max': 0.05,
    'kl_n_cycles': 3,                # v4: 3 cycles (was 2) for 80 epochs
    'kl_ratio': 0.5,

    # Free bits
    'free_bits': 0.20,               # v4: slightly lower than v3 (0.25) for tighter reconstruction

    # Loss weighting
    'ciede_weight': 1.0,
    'mse_weight': 0.2,

    # Data
    'image_size': 256,
    'num_workers': 2,
    'seed': 42,

    # Generative validation (metric evaluation on sampled palettes)
    'gen_val_every': 5,              # run generative validation every N epochs
    'gen_val_n_samples': 100,        # basemaps to sample for generative validation

    # Checkpointing
    'checkpoint_dir': '/content/drive/MyDrive/CartoPalette/checkpoints_v4',
    'save_every': 5,
}

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(CONFIG['seed'])

print('Config v4 loaded.')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')

## 3. Dataset

In [ ]:
# ── Encoding maps ──
# v4: qualitative removed — 2-slot one-hot (sequential=0, diverging=1)
SCHEME_TO_IDX = {'sequential': 0, 'diverging': 1}
SCALE_TO_IDX = {'overview': 0, 'regional': 1, 'local': 2}
NCLASSES_TO_IDX = {3: 0, 4: 1, 5: 2, 7: 3, 9: 4}


class CartoPaletteDataset(Dataset):
    """PyTorch Dataset for CartoPalette v4 training.

    Each sample: (basemap_image, condition_vector, palette_lab, n_classes)
    
    v4 changes:
    - Only sequential and diverging (no qualitative)
    - Labels generated with 80% basemap-dependent scoring
    - Diversity-aware top-k selection in ground truth
    """

    def __init__(self, labels_df, basemaps_dir, split='train', transform=None):
        self.basemaps_dir = Path(basemaps_dir)
        self.transform = transform

        # Filter to split and positive labels only for generation training
        self.data = labels_df[
            (labels_df['split'] == split) & (labels_df['label_type'] == 'positive')
        ].reset_index(drop=True)

        # Verify: no qualitative scheme types
        scheme_types = self.data['scheme_type'].unique()
        assert all(s in SCHEME_TO_IDX for s in scheme_types), \
            f'Unexpected scheme types: {scheme_types}'

        print(f'  {split}: {len(self.data)} samples')
        print(f'    Scheme distribution: {self.data["scheme_type"].value_counts().to_dict()}')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # ── Load basemap image ──
        img_path = self.basemaps_dir / f"{row['patch_id']}.png"
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # ── Parse palette (stored as JSON string) ──
        palette_lab = np.array(json.loads(row['palette_lab']), dtype=np.float32)
        n_colors = len(palette_lab)

        # Normalize L* to [0,1], a* and b* to ~[-1,1]
        palette_norm = palette_lab.copy()
        palette_norm[:, 0] /= 100.0        # L*: [0,100] → [0,1]
        palette_norm[:, 1] /= 128.0        # a*: [-128,127] → ~[-1,1]
        palette_norm[:, 2] /= 128.0        # b*: [-128,127] → ~[-1,1]

        # Pad to max_palette_size (9 colors)
        padded = np.zeros((9, 3), dtype=np.float32)
        padded[:n_colors] = palette_norm
        palette_flat = padded.flatten()  # (27,)

        # ── Build condition vector (11 dims, same as v3 for compat) ──
        # n_classes one-hot (5 values: 3,4,5,7,9)
        n_classes_oh = np.zeros(5, dtype=np.float32)
        n_classes_oh[NCLASSES_TO_IDX[row['n_classes']]] = 1.0

        # scheme_type one-hot (2 slots: sequential=0, diverging=1)
        scheme_oh = np.zeros(2, dtype=np.float32)
        scheme_oh[SCHEME_TO_IDX[row['scheme_type']]] = 1.0

        # scale_class one-hot (3 values)
        scale_oh = np.zeros(3, dtype=np.float32)
        scale_oh[SCALE_TO_IDX[row['scale_class']]] = 1.0

        condition_meta = np.concatenate([n_classes_oh, scheme_oh, scale_oh])  # (10,)

        return (
            image,
            torch.tensor(condition_meta),
            torch.tensor(palette_flat),
            torch.tensor(n_colors, dtype=torch.long),
        )


# ── Transforms ──
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# v4: NO ColorJitter! Geometric augmentations only.
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Dataset class defined.')

In [ ]:
# ── Load data ──
print('Loading labels...')
labels_df = pd.read_csv(labels_path)
print(f'Total labels: {len(labels_df)}')
print(f'Columns: {list(labels_df.columns)}')
print()

# v4 sanity checks
assert 'lightness_contrast' in labels_df.columns, 'Missing v4 metric: lightness_contrast'
assert 'hue_contrast' in labels_df.columns, 'Missing v4 metric: hue_contrast'
scheme_types = labels_df['scheme_type'].unique()
print(f'Scheme types in data: {scheme_types}')
assert 'qualitative' not in scheme_types, 'v4 data should not contain qualitative!'
print()

# Show v4 metric statistics for positive labels
pos = labels_df[labels_df['label_type'] == 'positive']
v4_metrics = ['composite_score', 'basemap_contrast', 'lightness_contrast',
              'hue_contrast', 'distinguishability', 'cvd_robustness', 'perceptual_ordering']
for m in v4_metrics:
    if m in pos.columns:
        print(f'  {m}: mean={pos[m].mean():.3f}, std={pos[m].std():.3f}, '
              f'min={pos[m].min():.3f}, max={pos[m].max():.3f}')
print()

# Create datasets
print('Creating datasets:')
train_dataset = CartoPaletteDataset(labels_df, basemaps_dir, split='train', transform=train_transform)
val_dataset = CartoPaletteDataset(labels_df, basemaps_dir, split='val', transform=val_transform)
test_dataset = CartoPaletteDataset(labels_df, basemaps_dir, split='test', transform=val_transform)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'], pin_memory=True,
)
test_loader = DataLoader(
    test_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'], pin_memory=True,
)

print(f'\nTrain batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

## 4. Model Architecture

Identical to v3 — the improvement comes from the data, not the architecture.

In [ ]:
class BasemapEncoder(nn.Module):
    """EfficientNet-B0 with projection head for basemap-discriminative features.

    Blocks 0-2 frozen (basic edges/textures).
    Blocks 3-8 trainable to learn map-specific features.
    Projection head amplifies basemap differences.
    """

    def __init__(self, projection_dim=256):
        super().__init__()
        efficientnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

        self.features = efficientnet.features
        self.pool = efficientnet.avgpool

        # Freeze blocks 0-2 (very low-level features)
        for i, block in enumerate(self.features):
            if i < 3:
                for param in block.parameters():
                    param.requires_grad = False

        # Projection head: 1280 → projection_dim
        self.projection = nn.Sequential(
            nn.Linear(1280, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, projection_dim),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = x.flatten(1)          # (batch, 1280)
        x = self.projection(x)    # (batch, projection_dim)
        return x


class CVAEEncoder(nn.Module):
    """CVAE encoder: (condition, palette) -> (mu, logvar)."""

    def __init__(self, condition_dim, palette_dim, hidden_dim, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(condition_dim + palette_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, condition, palette):
        x = torch.cat([condition, palette], dim=-1)
        h = self.net(x)
        return self.fc_mu(h), self.fc_logvar(h)


class CVAEDecoder(nn.Module):
    """CVAE decoder: (condition, z) -> reconstructed palette."""

    def __init__(self, condition_dim, latent_dim, hidden_dim, palette_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(condition_dim + latent_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, palette_dim),
        )

    def forward(self, condition, z):
        return self.net(torch.cat([condition, z], dim=-1))


class ChromaMapCVAE(nn.Module):
    """CartoPalette v4 CVAE (class name kept for checkpoint compatibility).

    Basemap → Projected CNN features → CVAE → Palette.
    Condition = [projected_features (256) + metadata_one_hots (11)] = 267 dims.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.latent_dim = config['latent_dim']

        self.cnn = BasemapEncoder(projection_dim=config['cnn_projection_dim'])
        self.metadata_scale = nn.Parameter(torch.ones(10) * 1.0)

        condition_dim = config['condition_dim']
        self.encoder = CVAEEncoder(
            condition_dim, config['palette_dim'],
            config['hidden_dim'], config['latent_dim'],
        )
        self.decoder = CVAEDecoder(
            condition_dim, config['latent_dim'],
            config['hidden_dim'], config['palette_dim'],
        )

    def build_condition(self, cnn_features, metadata):
        scaled_meta = metadata * self.metadata_scale
        return torch.cat([cnn_features, scaled_meta], dim=-1)

    def forward(self, image, metadata, palette):
        cnn_features = self.cnn(image)
        condition = self.build_condition(cnn_features, metadata)
        mu, logvar = self.encoder(condition, palette)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(condition, z)
        return recon, mu, logvar, cnn_features

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + torch.randn_like(std) * std
        return mu

    @torch.no_grad()
    def generate(self, image, metadata, n_samples=5):
        self.eval()
        cnn_features = self.cnn(image)
        condition = self.build_condition(cnn_features, metadata)
        condition = condition.repeat(n_samples, 1)
        z = torch.randn(n_samples, self.latent_dim, device=image.device)
        palettes = self.decoder(condition, z)
        return palettes.view(n_samples, 9, 3)


# ── Instantiate v4 ──
model = ChromaMapCVAE(CONFIG).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Frozen (CNN blocks 0-2): {total_params - trainable_params:,}')

# Quick feature check
print('\n--- Quick feature check ---')
model.eval()
with torch.no_grad():
    dummy1 = torch.randn(1, 3, 256, 256, device=device)
    dummy2 = torch.randn(1, 3, 256, 256, device=device)
    f1 = model.cnn(dummy1).cpu().numpy().flatten()
    f2 = model.cnn(dummy2).cpu().numpy().flatten()
    cos = np.dot(f1, f2) / (np.linalg.norm(f1) * np.linalg.norm(f2) + 1e-8)
    print(f'Random images cosine sim: {cos:.4f} (should be << 0.99)')
    print(f'Feature norm: {np.linalg.norm(f1):.2f} (should be >> 1.13)')

## 5. Loss Function & Training

In [ ]:
# ══════════════════════════════════════════════════
# Differentiable CIEDE2000 Loss (PyTorch)
# ══════════════════════════════════════════════════

def _deg2rad(deg):
    return deg * (math.pi / 180.0)

def _rad2deg(rad):
    return rad * (180.0 / math.pi)

def ciede2000_loss(lab1, lab2, mask=None):
    """Differentiable CIEDE2000 color difference loss.

    Args:
        lab1: (batch, n_colors, 3) predicted CIELAB (denormalized)
        lab2: (batch, n_colors, 3) target CIELAB (denormalized)
        mask: (batch, n_colors, 1) binary mask for valid colors

    Returns:
        Scalar mean CIEDE2000 distance over valid colors.
    """
    eps = 1e-8

    L1, a1, b1 = lab1[..., 0], lab1[..., 1], lab1[..., 2]
    L2, a2, b2 = lab2[..., 0], lab2[..., 1], lab2[..., 2]

    # Step 1: Calculate C'ab and h'ab
    C1 = torch.sqrt(a1**2 + b1**2 + eps)
    C2 = torch.sqrt(a2**2 + b2**2 + eps)
    C_avg = (C1 + C2) / 2.0
    C_avg7 = C_avg**7
    G = 0.5 * (1.0 - torch.sqrt(C_avg7 / (C_avg7 + 25.0**7 + eps)))

    a1p = a1 * (1.0 + G)
    a2p = a2 * (1.0 + G)

    C1p = torch.sqrt(a1p**2 + b1**2 + eps)
    C2p = torch.sqrt(a2p**2 + b2**2 + eps)

    h1p = _rad2deg(torch.atan2(b1, a1p + eps)) % 360.0
    h2p = _rad2deg(torch.atan2(b2, a2p + eps)) % 360.0

    # Step 2: Delta values
    dLp = L2 - L1
    dCp = C2p - C1p

    dhp_cond1 = (h2p - h1p).abs() <= 180.0
    dhp_cond2 = h2p - h1p > 180.0

    dhp = torch.where(dhp_cond1, h2p - h1p,
           torch.where(dhp_cond2, h2p - h1p - 360.0, h2p - h1p + 360.0))

    dHp = 2.0 * torch.sqrt(C1p * C2p + eps) * torch.sin(_deg2rad(dhp / 2.0))

    # Step 3: Weighting functions
    Lp_avg = (L1 + L2) / 2.0
    Cp_avg = (C1p + C2p) / 2.0

    hp_sum = h1p + h2p
    hp_diff_abs = (h1p - h2p).abs()
    hp_avg = torch.where(
        hp_diff_abs <= 180.0,
        hp_sum / 2.0,
        torch.where(hp_sum < 360.0, (hp_sum + 360.0) / 2.0, (hp_sum - 360.0) / 2.0)
    )

    T = (1.0
         - 0.17 * torch.cos(_deg2rad(hp_avg - 30.0))
         + 0.24 * torch.cos(_deg2rad(2.0 * hp_avg))
         + 0.32 * torch.cos(_deg2rad(3.0 * hp_avg + 6.0))
         - 0.20 * torch.cos(_deg2rad(4.0 * hp_avg - 63.0)))

    SL = 1.0 + 0.015 * (Lp_avg - 50.0)**2 / torch.sqrt(20.0 + (Lp_avg - 50.0)**2 + eps)
    SC = 1.0 + 0.045 * Cp_avg
    SH = 1.0 + 0.015 * Cp_avg * T

    Cp_avg7 = Cp_avg**7
    RT = (-2.0 * torch.sqrt(Cp_avg7 / (Cp_avg7 + 25.0**7 + eps))
          * torch.sin(_deg2rad(60.0 * torch.exp(-((hp_avg - 275.0) / 25.0)**2))))

    dE = torch.sqrt(
        (dLp / SL)**2 + (dCp / SC)**2 + (dHp / SH)**2
        + RT * (dCp / SC) * (dHp / SH) + eps
    )

    if mask is not None:
        dE = dE * mask.squeeze(-1)
        return dE.sum() / (mask.sum() + eps)
    else:
        return dE.mean()


def get_kl_weight_cyclical(epoch, n_epochs, n_cycles, ratio, max_weight):
    """Cyclical KL annealing schedule."""
    cycle_length = n_epochs / n_cycles
    tau = ((epoch - 1) % cycle_length) / cycle_length
    if tau < ratio:
        return max_weight * (tau / ratio)
    else:
        return max_weight


def cvae_loss(recon, target, mu, logvar, n_colors, kl_weight, config, max_colors=9):
    """CVAE loss = CIEDE2000 + MSE + KL with free bits."""
    batch_size = target.size(0)

    recon_colors = recon.view(batch_size, max_colors, 3)
    target_colors = target.view(batch_size, max_colors, 3)

    # Mask for actual colors (not padding)
    mask = torch.arange(max_colors, device=target.device).unsqueeze(0) < n_colors.unsqueeze(1)
    mask = mask.unsqueeze(-1).float()  # (batch, max_colors, 1)

    # ── CIEDE2000 loss (denormalize to CIELAB first) ──
    recon_lab = recon_colors.clone()
    target_lab = target_colors.clone()
    recon_lab[..., 0] = recon_lab[..., 0] * 100.0
    recon_lab[..., 1] = recon_lab[..., 1] * 128.0
    recon_lab[..., 2] = recon_lab[..., 2] * 128.0
    target_lab[..., 0] = target_lab[..., 0] * 100.0
    target_lab[..., 1] = target_lab[..., 1] * 128.0
    target_lab[..., 2] = target_lab[..., 2] * 128.0

    ciede_loss = ciede2000_loss(recon_lab, target_lab, mask)

    # ── MSE loss (normalized space, for gradient stability) ──
    sq_diff = (recon_colors - target_colors) ** 2 * mask
    mse_loss = sq_diff.sum() / (mask.sum() + 1e-8)

    # ── KL divergence with free bits ──
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    free_bits = config.get('free_bits', 0.5)
    kl_per_dim = torch.clamp(kl_per_dim, min=free_bits)
    kl_loss = kl_per_dim.sum(dim=-1).mean()

    # ── Combine ──
    ciede_normalized = ciede_loss / 50.0
    recon_combined = config['ciede_weight'] * ciede_normalized + config['mse_weight'] * mse_loss
    total = recon_combined + kl_weight * kl_loss

    return total, recon_combined.item(), kl_loss.item(), ciede_loss.item()


print('Loss function defined (CIEDE2000 + MSE + Cyclical KL + Free Bits).')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, scaler, epoch, config):
    model.train()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    total_ciede = 0
    n_batches = 0

    kl_weight = get_kl_weight_cyclical(
        epoch, config['epochs'], config['kl_n_cycles'],
        config['kl_ratio'], config['kl_weight_max']
    )

    for batch_idx, (images, metadata, palettes, n_colors) in enumerate(loader):
        images = images.to(device)
        metadata = metadata.to(device)
        palettes = palettes.to(device)
        n_colors = n_colors.to(device)

        optimizer.zero_grad()

        with autocast(device_type='cuda'):
            recon, mu, logvar, cnn_feats = model(images, metadata, palettes)
            loss, recon_l, kl_l, ciede_l = cvae_loss(
                recon, palettes, mu, logvar, n_colors, kl_weight, config
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        total_recon += recon_l
        total_kl += kl_l
        total_ciede += ciede_l
        n_batches += 1

        if batch_idx % 100 == 0:
            print(f'  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f} '
                  f'(recon: {recon_l:.4f}, kl: {kl_l:.4f}, dE: {ciede_l:.2f}) '
                  f'| kl_w: {kl_weight:.4f}')

    scheduler.step()

    return {
        'loss': total_loss / n_batches,
        'recon': total_recon / n_batches,
        'kl': total_kl / n_batches,
        'ciede': total_ciede / n_batches,
        'kl_weight': kl_weight,
        'lr_cnn': optimizer.param_groups[0]['lr'],
        'lr_main': optimizer.param_groups[1]['lr'],
    }


@torch.no_grad()
def validate(model, loader, epoch, config):
    model.eval()
    total_loss = 0
    total_recon = 0
    total_kl = 0
    total_ciede = 0
    n_batches = 0

    kl_weight = get_kl_weight_cyclical(
        epoch, config['epochs'], config['kl_n_cycles'],
        config['kl_ratio'], config['kl_weight_max']
    )

    for images, metadata, palettes, n_colors in loader:
        images = images.to(device)
        metadata = metadata.to(device)
        palettes = palettes.to(device)
        n_colors = n_colors.to(device)

        with autocast(device_type='cuda'):
            recon, mu, logvar, cnn_feats = model(images, metadata, palettes)
            loss, recon_l, kl_l, ciede_l = cvae_loss(
                recon, palettes, mu, logvar, n_colors, kl_weight, config
            )

        total_loss += loss.item()
        total_recon += recon_l
        total_kl += kl_l
        total_ciede += ciede_l
        n_batches += 1

    return {
        'loss': total_loss / n_batches,
        'recon': total_recon / n_batches,
        'kl': total_kl / n_batches,
        'ciede': total_ciede / n_batches,
    }


def save_checkpoint(model, optimizer, scheduler, scaler, epoch, metrics, config):
    path = os.path.join(config['checkpoint_dir'], f'checkpoint_epoch_{epoch:03d}.pt')
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'metrics': metrics,
        'config': config,
    }, path)
    print(f'  -> Checkpoint saved: {path}')


def load_checkpoint(model, optimizer, scheduler, scaler, config):
    """Resume from latest checkpoint if available."""
    cp_dir = config['checkpoint_dir']
    if not os.path.exists(cp_dir):
        return 0, []
    checkpoints = sorted([f for f in os.listdir(cp_dir) if f.startswith('checkpoint_') and f.endswith('.pt')])
    if not checkpoints:
        return 0, []

    latest = os.path.join(cp_dir, checkpoints[-1])
    print(f'Resuming from: {latest}')

    checkpoint = torch.load(latest, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])

    return checkpoint['epoch'], checkpoint.get('metrics', [])


print('Training functions defined.')

## 6. Train!

In [ ]:
# ══════════════════════════════════════════════════════════════
# LOAD BEST GENERATIVE MODEL (skip training)
# ══════════════════════════════════════════════════════════════
# Use this cell INSTEAD of the training loop when you already have
# a trained best_model_generative.pt and just want to evaluate.
# Run cells 0-15 first (setup, config, data, model architecture),
# then run THIS cell, then skip to cell 18+ for evaluation.
# ══════════════════════════════════════════════════════════════

from research.data_pipeline.score_palettes import compute_composite_score

GEN_VAL_WEIGHTS = {
    'distinguishability': 0.10,
    'basemap_contrast': 0.20,
    'lightness_contrast': 0.30,
    'hue_contrast': 0.30,
    'cvd_robustness': 0.05,
    'perceptual_ordering': 0.05,
}

IDX_TO_SCHEME_GEN = {v: k for k, v in SCHEME_TO_IDX.items()}
IDX_TO_NCLASSES_GEN = {v: k for k, v in NCLASSES_TO_IDX.items()}

# ── Load checkpoint ──
ckpt_path = os.path.join(CONFIG['checkpoint_dir'], 'best_model_generative.pt')
assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# ── Print info ──
ep = ckpt.get('epoch', '?')
gen_val = ckpt.get('gen_val', {})
tm = gen_val.get('tool_metric', gen_val.get('composite_score', 0))
bok = gen_val.get('best_of_k', {})
p70 = gen_val.get('prob_above_070', 0)
dist = gen_val.get('mean_best_sample_distinguishability', 0)

print(f"Loaded best_model_generative.pt (epoch {ep})")
print(f"  tool_metric (best-of-20): {tm:.4f}")
for k, v in sorted(bok.items(), key=lambda x: int(x[0])):
    print(f"  best-of-{int(k):>2}: {v:.4f}")
print(f"  P(>=0.70): {p70:.2%}")
print(f"  distinguishability: {dist:.4f}")
print(f"\nModel is in eval mode. Ready for evaluation cells.")


In [ ]:
# ══════════════════════════════════════════════════════════════
# TRAINING LOOP with Dual Early Stopping + Generative Validation
# ══════════════════════════════════════════════════════════════
# v4.1: Identical hyperparameters, only training data changed (dark-bias fixes).
#
# Early stopping: two independent counters
#   - recon_patience: val recon loss hasn't improved for N epochs
#   - gen_patience:   gen_val tool_metric hasn't improved for M checks
# Training stops ONLY when BOTH are exhausted simultaneously.
#
# Key: Early stopping tracks best-of-20 (tool_metric), NOT best-of-1.
# This is the metric closest to actual tool performance (sample+rerank).
# ══════════════════════════════════════════════════════════════

from research.data_pipeline.score_palettes import compute_composite_score

GEN_VAL_WEIGHTS = {
    'distinguishability': 0.10,
    'basemap_contrast': 0.20,
    'lightness_contrast': 0.30,
    'hue_contrast': 0.30,
    'cvd_robustness': 0.05,
    'perceptual_ordering': 0.05,
}

IDX_TO_SCHEME_GEN = {v: k for k, v in SCHEME_TO_IDX.items()}
IDX_TO_NCLASSES_GEN = {v: k for k, v in NCLASSES_TO_IDX.items()}


@torch.no_grad()
def generative_validation(model, dataset, config, n_basemaps=None, k_values=[1, 5, 10, 20]):
    """Sample palettes, score them, and compute best-of-k metrics.

    IMPORTANT: best-of-k uses the GENERATION ORDER of samples (iid draws),
    NOT a pre-sorted order. This correctly measures how many samples you
    need to draw to find a good palette.

    Returns dict with:
      - composite_score: mean best-of-1 composite (backward compat)
      - tool_metric: mean best-of-20 composite (primary early stopping metric)
      - best_of_k: {k: mean_best_composite} for each k in k_values
      - mean_best_sample_distinguishability: mean best-of-1 distinguishability
      - prob_above_070: P(best-of-20 composite >= 0.70)
    """
    model.eval()
    if n_basemaps is None:
        n_basemaps = config.get('gen_val_n_samples', 100)

    indices = np.random.choice(len(dataset), min(n_basemaps, len(dataset)), replace=False)
    max_k = max(k_values)

    best_of_k_composites = {k: [] for k in k_values}
    best_sample_dists = []
    above_070_count = 0

    for idx in indices:
        img_tensor, metadata, palette_flat, n_colors = dataset[idx]
        n_col = n_colors.item()
        row = dataset.data.iloc[idx]
        patch_id = row['patch_id']
        scheme_type = row['scheme_type']

        # Load basemap colors
        color_path = os.path.join(basemap_colors_dir, f'{patch_id}_colors.json')
        if not os.path.exists(color_path):
            continue
        with open(color_path) as f:
            bm_colors = np.array(json.load(f)['colors_lab'])

        # Generate max_k samples
        img_batch = img_tensor.unsqueeze(0).to(device)
        meta_batch = metadata.unsqueeze(0).to(device)
        generated = model.generate(img_batch, meta_batch, n_samples=max_k)
        generated = generated.cpu().numpy()

        # Score all samples -- keep in GENERATION ORDER (do NOT sort!)
        composites_in_order = []
        dists_in_order = []
        for s in range(max_k):
            pal_lab = generated[s].copy()
            pal_lab[:, 0] *= 100.0
            pal_lab[:, 1] *= 128.0
            pal_lab[:, 2] *= 128.0
            pal_lab[:, 0] = np.clip(pal_lab[:, 0], 0, 100)
            pal_lab[:, 1] = np.clip(pal_lab[:, 1], -128, 127)
            pal_lab[:, 2] = np.clip(pal_lab[:, 2], -128, 127)
            pal_lab = pal_lab[:n_col]

            sc = compute_composite_score(pal_lab, bm_colors, scheme_type, GEN_VAL_WEIGHTS)
            composites_in_order.append(sc['composite_score'])
            dists_in_order.append(sc['distinguishability'])

        # Best-of-k: max of the FIRST k samples (iid generation order)
        for k in k_values:
            best_of_k_composites[k].append(max(composites_in_order[:k]))

        # Distinguishability of the globally best sample
        best_idx = composites_in_order.index(max(composites_in_order))
        best_sample_dists.append(dists_in_order[best_idx])

        # P(best-of-max_k >= 0.70)
        if max(composites_in_order) >= 0.70:
            above_070_count += 1

    n_valid = len(best_of_k_composites[k_values[0]])
    if n_valid == 0:
        return {'composite_score': 0.0, 'tool_metric': 0.0, 'best_of_k': {},
                'mean_best_sample_distinguishability': 0.0, 'prob_above_070': 0.0}

    # tool_metric = best-of-20 (primary early stopping metric)
    tool_k = 20 if 20 in best_of_k_composites else max(k_values)
    tool_metric = float(np.mean(best_of_k_composites[tool_k]))

    result = {
        'composite_score': float(np.mean(best_of_k_composites.get(1, best_of_k_composites[k_values[0]]))),
        'tool_metric': tool_metric,
        'best_of_k': {k: float(np.mean(v)) for k, v in best_of_k_composites.items()},
        'mean_best_sample_distinguishability': float(np.mean(best_sample_dists)),
        'prob_above_070': above_070_count / n_valid,
        'n_evaluated': n_valid,
    }

    return result


# == Setup ==
optimizer = torch.optim.AdamW([
    {'params': model.cnn.parameters(), 'lr': CONFIG['lr_cnn']},
    {'params': list(model.encoder.parameters()) +
               list(model.decoder.parameters()) +
               [model.metadata_scale],
     'lr': CONFIG['lr_cvae']},
], weight_decay=CONFIG['weight_decay'])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs'], eta_min=CONFIG['lr_min']
)
scaler = GradScaler()

# == Resume from checkpoint if available ==
start_epoch, history = load_checkpoint(model, optimizer, scheduler, scaler, CONFIG)
print(f'Starting from epoch {start_epoch + 1}')

# == Early stopping state ==
# gen_patience now tracks tool_metric (best-of-20), not best-of-1
best_val_recon = float('inf')
best_tool_metric = 0.0
recon_patience_counter = 0
gen_patience_counter = 0

# Restore patience state from history
for h in history:
    val_recon = h['val']['recon']
    if val_recon < best_val_recon:
        best_val_recon = val_recon
        recon_patience_counter = 0
    else:
        recon_patience_counter += 1

    if 'gen_val' in h:
        # Use tool_metric if available, fall back to composite_score for old checkpoints
        tm = h['gen_val'].get('tool_metric', h['gen_val'].get('composite_score', 0))
        if tm > best_tool_metric:
            best_tool_metric = tm
            gen_patience_counter = 0
        else:
            gen_patience_counter += 1

print(f'Restored patience: recon={recon_patience_counter}/{CONFIG["recon_patience"]}, '
      f'gen={gen_patience_counter}/{CONFIG["gen_patience"]}')
print(f'Best so far: val_recon={best_val_recon:.4f}, tool_metric(best-of-20)={best_tool_metric:.4f}')

# == Training loop ==
for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
    print(f'\n{"="*60}')
    print(f'Epoch {epoch}/{CONFIG["epochs"]}')
    print(f'{"="*60}')

    # Train
    train_metrics = train_epoch(model, train_loader, optimizer, scheduler, scaler, epoch, CONFIG)

    # Validate
    val_metrics = validate(model, val_loader, epoch, CONFIG)

    epoch_log = {
        'epoch': epoch,
        'train': train_metrics,
        'val': val_metrics,
    }

    # == Recon early stopping ==
    if val_metrics['recon'] < best_val_recon:
        best_val_recon = val_metrics['recon']
        recon_patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_recon': val_metrics['recon'],
        }, os.path.join(CONFIG['checkpoint_dir'], 'best_model.pt'))
        print(f'  ** New best val recon: {val_metrics["recon"]:.4f} -> saved best_model.pt')
    else:
        recon_patience_counter += 1

    # == Generative validation (early stopping on tool_metric = best-of-20) ==
    if epoch % CONFIG['gen_val_every'] == 0:
        print(f'\n  Running generative validation (n={CONFIG["gen_val_n_samples"]})...')
        gen_result = generative_validation(model, val_dataset, CONFIG)
        epoch_log['gen_val'] = gen_result

        tm = gen_result['tool_metric']
        print(f'  Gen best-of-1:  {gen_result["composite_score"]:.4f}')
        for k, v in gen_result['best_of_k'].items():
            marker = ' <-- early stop metric' if k == 20 else ''
            print(f'  Gen best-of-{k:>2}: {v:.4f}{marker}')
        print(f'  Gen distinguishability:  {gen_result["mean_best_sample_distinguishability"]:.4f}')
        print(f'  P(best-of-20 >= 0.70):   {gen_result["prob_above_070"]:.2%}')

        if tm > best_tool_metric:
            best_tool_metric = tm
            gen_patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'gen_val': gen_result,
            }, os.path.join(CONFIG['checkpoint_dir'], 'best_model_generative.pt'))
            print(f'  ** New best tool_metric (best-of-20): {tm:.4f} -> saved best_model_generative.pt')
        else:
            gen_patience_counter += 1

    history.append(epoch_log)

    # == Periodic checkpoint ==
    if epoch % CONFIG['save_every'] == 0:
        save_checkpoint(model, optimizer, scheduler, scaler, epoch, history, CONFIG)

    # == Status ==
    print(f'\n  val_recon={val_metrics["recon"]:.4f} | '
          f'patience: recon={recon_patience_counter}/{CONFIG["recon_patience"]}, '
          f'gen={gen_patience_counter}/{CONFIG["gen_patience"]}')

    # == Dual early stopping ==
    if recon_patience_counter >= CONFIG['recon_patience'] and gen_patience_counter >= CONFIG['gen_patience']:
        print(f'\n  *** DUAL EARLY STOP at epoch {epoch} ***')
        print(f'      Recon stalled for {recon_patience_counter} epochs, gen stalled for {gen_patience_counter} checks.')
        save_checkpoint(model, optimizer, scheduler, scaler, epoch, history, CONFIG)
        break

# Final checkpoint
save_checkpoint(model, optimizer, scheduler, scaler, epoch, history, CONFIG)
print(f'\nTraining complete at epoch {epoch}.')
print(f'Best val recon: {best_val_recon:.4f}')
print(f'Best tool_metric (best-of-20): {best_tool_metric:.4f}')

## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_list = [h['epoch'] for h in history]
train_loss = [h['train']['loss'] for h in history]
val_loss = [h['val']['loss'] for h in history]
train_recon = [h['train']['recon'] for h in history]
val_recon = [h['val']['recon'] for h in history]
train_kl = [h['train']['kl'] for h in history]
val_kl = [h['val']['kl'] for h in history]
train_ciede = [h['train'].get('ciede', 0) for h in history]
val_ciede = [h['val'].get('ciede', 0) for h in history]
kl_weights = [h['train']['kl_weight'] for h in history]

fig, axes = plt.subplots(2, 4, figsize=(24, 10))

axes[0,0].plot(epochs_list, train_loss, label='Train', linewidth=2)
axes[0,0].plot(epochs_list, val_loss, label='Val', linewidth=2)
axes[0,0].set_title('Total Loss', fontsize=14)
axes[0,0].set_xlabel('Epoch')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(epochs_list, train_recon, label='Train', linewidth=2)
axes[0,1].plot(epochs_list, val_recon, label='Val', linewidth=2)
axes[0,1].set_title('Reconstruction Loss', fontsize=14)
axes[0,1].set_xlabel('Epoch')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(epochs_list, train_kl, label='Train', linewidth=2)
axes[0,2].plot(epochs_list, val_kl, label='Val', linewidth=2)
axes[0,2].set_title('KL Divergence', fontsize=14)
axes[0,2].set_xlabel('Epoch')
axes[0,2].legend()
axes[0,2].grid(True, alpha=0.3)

axes[0,3].plot(epochs_list, train_ciede, label='Train', linewidth=2)
axes[0,3].plot(epochs_list, val_ciede, label='Val', linewidth=2)
axes[0,3].set_title('Mean CIEDE2000 (dE)', fontsize=14)
axes[0,3].set_xlabel('Epoch')
axes[0,3].legend()
axes[0,3].grid(True, alpha=0.3)

axes[1,0].plot(epochs_list, kl_weights, linewidth=2, color='red')
axes[1,0].set_title('KL Weight (Cyclical)', fontsize=14)
axes[1,0].set_xlabel('Epoch')
axes[1,0].grid(True, alpha=0.3)

# Both learning rates
lrs_cnn = [h['train']['lr_cnn'] for h in history]
lrs_main = [h['train']['lr_main'] for h in history]
axes[1,1].plot(epochs_list, lrs_cnn, label='CNN (blocks 3-8)', linewidth=2, color='#2196F3')
axes[1,1].plot(epochs_list, lrs_main, label='CVAE + projection', linewidth=2, color='#4CAF50')
axes[1,1].set_title('Learning Rates', fontsize=14)
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_yscale('log')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# Generative validation composite score over time
gen_epochs = [h['epoch'] for h in history if 'gen_val' in h]
gen_composites = [h['gen_val']['composite_score'] for h in history if 'gen_val' in h]
if gen_epochs:
    axes[1,2].plot(gen_epochs, gen_composites, 'o-', linewidth=2, color='#FF9800', markersize=6)
    axes[1,2].set_title('Gen. Composite Score (val)', fontsize=14)
    axes[1,2].set_xlabel('Epoch')
    axes[1,2].set_ylim(0, 1)
    axes[1,2].grid(True, alpha=0.3)
    # Add horizontal line for ground-truth mean
    gt_mean = np.mean([h['gen_val'].get('composite_score', 0) for h in history if 'gen_val' in h])
    axes[1,2].axhline(y=gt_mean, color='gray', linestyle='--', alpha=0.5, label=f'Mean: {gt_mean:.3f}')
    axes[1,2].legend()
else:
    axes[1,2].text(0.5, 0.5, 'No gen_val data', ha='center', va='center', transform=axes[1,2].transAxes)
    axes[1,2].set_title('Gen. Composite Score', fontsize=14)

# Per-metric breakdown from latest gen_val
latest_gen = [h for h in history if 'gen_val' in h]
if latest_gen:
    latest = latest_gen[-1]['gen_val']
    metric_names = ['basemap_contrast', 'lightness_contrast', 'hue_contrast',
                    'distinguishability', 'cvd_robustness', 'perceptual_ordering']
    metric_labels = ['Contrast', 'L*', 'Hue', 'Distin.', 'CVD', 'Order']
    vals = [latest.get(m, 0) for m in metric_names]
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336', '#607D8B']
    bars = axes[1,3].bar(metric_labels, vals, color=colors, alpha=0.8)
    axes[1,3].set_title(f'Gen. Metrics (epoch {latest_gen[-1]["epoch"]})', fontsize=14)
    axes[1,3].set_ylim(0, 1)
    axes[1,3].grid(True, alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        axes[1,3].text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.2f}',
                       ha='center', fontsize=9)
else:
    axes[1,3].text(0.5, 0.5, 'No gen_val data', ha='center', va='center', transform=axes[1,3].transAxes)
    axes[1,3].set_title('Gen. Metrics', fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['checkpoint_dir'], 'training_curves_v4.png'), dpi=150,
            bbox_inches='tight')
plt.show()
print('Training curves saved.')

## 8. Evaluation & Demo

In [ ]:
def denormalize_palette(palette_norm, n_colors):    """Convert normalized palette back to CIELAB."""    palette = palette_norm.copy()    palette[:, 0] *= 100.0    # L*    palette[:, 1] *= 128.0    # a*    palette[:, 2] *= 128.0    # b*    palette[:, 0] = np.clip(palette[:, 0], 0, 100)    palette[:, 1] = np.clip(palette[:, 1], -128, 127)    palette[:, 2] = np.clip(palette[:, 2], -128, 127)    return palette[:n_colors]def lab_to_rgb_simple(lab):    """Simple LAB to RGB conversion for visualization."""    from skimage import color as skcolor    rgb = skcolor.lab2rgb(lab.reshape(1, -1, 3)).reshape(-1, 3)    return np.clip(rgb, 0, 1)# ═══════════════════════════════════════════════════════════════# MMR / Diversity-Aware Top-k Selection# ═══════════════════════════════════════════════════════════════def palette_distance(pal_a, pal_b):    """Mean per-color Euclidean distance in CIELAB between two palettes."""    return float(np.mean(np.sqrt(np.sum((pal_a - pal_b) ** 2, axis=1))))def mmr_top_k(scored_palettes, k=3, lambda_param=0.7):    """Maximal Marginal Relevance selection for diverse top-k palettes.    Args:        scored_palettes: list of (palette_lab, composite_score, score_dict)        k: number of palettes to select        lambda_param: 0=max diversity, 1=max relevance. 0.7 = relevance-heavy    Returns:        List of k selected (palette_lab, composite_score, score_dict) tuples    """    if len(scored_palettes) <= k:        return scored_palettes    # Normalize scores to [0, 1] for MMR    max_score = max(s[1] for s in scored_palettes)    min_score = min(s[1] for s in scored_palettes)    score_range = max(max_score - min_score, 1e-8)    selected = []    remaining = list(range(len(scored_palettes)))    # First pick: highest score    best_idx = max(remaining, key=lambda i: scored_palettes[i][1])    selected.append(best_idx)    remaining.remove(best_idx)    # Subsequent picks: MMR    for _ in range(k - 1):        if not remaining:            break        best_mmr = -float('inf')        best_r = None        for r in remaining:            # Relevance (normalized)            relevance = (scored_palettes[r][1] - min_score) / score_range            # Max similarity to already selected (distance-based)            max_sim = 0.0            for s_idx in selected:                dist = palette_distance(scored_palettes[r][0], scored_palettes[s_idx][0])                # Convert distance to similarity (closer = more similar)                sim = max(0, 1.0 - dist / 80.0)  # 80 dE = max expected distance                max_sim = max(max_sim, sim)            mmr = lambda_param * relevance - (1 - lambda_param) * max_sim            if mmr > best_mmr:                best_mmr = mmr                best_r = r        if best_r is not None:            selected.append(best_r)            remaining.remove(best_r)    return [scored_palettes[i] for i in selected]# ═══════════════════════════════════════════════════════════════# Sample + Rerank Evaluation Gallery# ═══════════════════════════════════════════════════════════════def visualize_sample_rerank(model, dataset, n_examples=8, n_raw=20, n_show=3):    """    Sample+Rerank evaluation with MMR diversity:      - For each basemap, generate n_raw palette samples      - Score ALL with the 6 cartographic metrics      - Select top n_show via MMR (diverse + high quality)      - Display: basemap | GT | MMR top-3 | worst    """    from research.data_pipeline.score_palettes import compute_composite_score    WEIGHTS = {        'distinguishability': 0.10,        'basemap_contrast': 0.20,        'lightness_contrast': 0.30,        'hue_contrast': 0.30,        'cvd_robustness': 0.05,        'perceptual_ordering': 0.05,    }    model.eval()    n_cols = 2 + n_show + 1  # basemap | GT | top1 | top2 | top3 | worst    fig, axes = plt.subplots(n_examples, n_cols,                             figsize=(3.5 * n_cols, 3.8 * n_examples))    indices = np.random.choice(len(dataset), n_examples, replace=False)    all_best_scores = []    all_worst_scores = []    all_mmr_scores = []    for row_idx, idx in enumerate(indices):        img_tensor, metadata, palette_flat, n_colors = dataset[idx]        n_col = n_colors.item()        data_row = dataset.data.iloc[idx]        patch_id = data_row['patch_id']        scheme_type = data_row['scheme_type']        # Load basemap colors        color_path = os.path.join(basemap_colors_dir, f'{patch_id}_colors.json')        if os.path.exists(color_path):            with open(color_path) as f:                bm_colors = np.array(json.load(f)['colors_lab'])        else:            bm_colors = None        # Show basemap        img_vis = img_tensor.numpy().transpose(1, 2, 0)        img_vis = img_vis * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)        img_vis = np.clip(img_vis, 0, 1)        axes[row_idx, 0].imshow(img_vis)        axes[row_idx, 0].set_title('Basemap', fontsize=9)        axes[row_idx, 0].axis('off')        # Show ground truth        gt_lab = denormalize_palette(palette_flat.numpy().reshape(9, 3), n_col)        gt_rgb = lab_to_rgb_simple(gt_lab)        gt_score = 0.0        if bm_colors is not None:            gt_scores_dict = compute_composite_score(gt_lab, bm_colors, scheme_type, WEIGHTS)            gt_score = gt_scores_dict['composite_score']        pal_img = np.ones((50, n_col * 50, 3))        for i in range(n_col):            pal_img[:, i*50:(i+1)*50] = gt_rgb[i]        axes[row_idx, 1].imshow(pal_img)        axes[row_idx, 1].set_title(f'GT ({gt_score:.2f})', fontsize=9, fontweight='bold')        axes[row_idx, 1].axis('off')        # Generate n_raw samples        img_batch = img_tensor.unsqueeze(0).to(device)        meta_batch = metadata.unsqueeze(0).to(device)        with torch.no_grad():            generated = model.generate(img_batch, meta_batch, n_samples=n_raw)        generated = generated.cpu().numpy()        # Score all samples        scored = []        for s in range(n_raw):            pal_lab = denormalize_palette(generated[s].copy(), n_col)            if bm_colors is not None:                sc = compute_composite_score(pal_lab, bm_colors, scheme_type, WEIGHTS)                scored.append((pal_lab, sc['composite_score'], sc))            else:                scored.append((pal_lab, 0.0, {}))        scored.sort(key=lambda x: x[1], reverse=True)        all_best_scores.append(scored[0][1])        all_worst_scores.append(scored[-1][1])        # MMR top-k selection        mmr_selected = mmr_top_k(scored, k=n_show, lambda_param=0.7)        all_mmr_scores.append(np.mean([s[1] for s in mmr_selected]))        # Show MMR top-k        for rank, (pal_lab, s_score, s_detail) in enumerate(mmr_selected):            pal_rgb = lab_to_rgb_simple(pal_lab)            pal_img = np.ones((50, n_col * 50, 3))            for i in range(n_col):                pal_img[:, i*50:(i+1)*50] = pal_rgb[i]            col = 2 + rank            axes[row_idx, col].imshow(pal_img)            dist = s_detail.get('distinguishability', 0)            axes[row_idx, col].set_title(                f'MMR#{rank+1} ({s_score:.2f}, d={dist:.2f})', fontsize=9,                color='green' if s_score >= gt_score * 0.9 else 'black')            axes[row_idx, col].axis('off')        # Show worst        w_pal_lab, w_score, w_detail = scored[-1]        w_rgb = lab_to_rgb_simple(w_pal_lab)        pal_img = np.ones((50, n_col * 50, 3))        for i in range(n_col):            pal_img[:, i*50:(i+1)*50] = w_rgb[i]        axes[row_idx, -1].imshow(pal_img)        axes[row_idx, -1].set_title(f'Worst ({w_score:.2f})', fontsize=9, color='red')        axes[row_idx, -1].axis('off')    fig.text(0.02, 0.99, f'Sample+Rerank+MMR: {n_raw} samples, MMR top-{n_show} (lambda=0.7) + worst',             fontsize=11, fontweight='bold', va='top')    plt.tight_layout(rect=[0, 0, 1, 0.98])    plt.savefig(os.path.join(CONFIG['checkpoint_dir'], 'eval_sample_rerank_mmr_v4.png'), dpi=150,                bbox_inches='tight')    plt.show()    print(f'\n{"="*60}')    print(f'Sample+Rerank+MMR Summary ({n_raw} samples per basemap)')    print(f'{"="*60}')    print(f'  Mean BEST composite:     {np.mean(all_best_scores):.4f}')    print(f'  Mean MMR-top-3 composite:{np.mean(all_mmr_scores):.4f}')    print(f'  Mean WORST composite:    {np.mean(all_worst_scores):.4f}')    print(f'  Spread (best-worst):     {np.mean(np.array(all_best_scores)-np.array(all_worst_scores)):.4f}')    print(f'{"="*60}')# ── Load best model ──gen_path = os.path.join(CONFIG['checkpoint_dir'], 'best_model_generative.pt')recon_path = os.path.join(CONFIG['checkpoint_dir'], 'best_model.pt')if os.path.exists(gen_path):    best_ckpt = torch.load(gen_path, map_location=device)    model.load_state_dict(best_ckpt['model_state_dict'])    gen_score = best_ckpt.get('gen_val', {}).get('composite_score', 0)    print(f'Loaded best GENERATIVE model from epoch {best_ckpt["epoch"]} (composite: {gen_score:.4f})')else:    best_ckpt = torch.load(recon_path, map_location=device)    model.load_state_dict(best_ckpt['model_state_dict'])    val_r = best_ckpt.get('val_recon', best_ckpt.get('val_loss', 0))    print(f'Loaded best RECON model from epoch {best_ckpt["epoch"]} (val_recon: {val_r:.4f})')# Install scikit-image for LAB to RGB!pip install scikit-image -q# Run sample+rerank+MMR evaluationvisualize_sample_rerank(model, test_dataset, n_examples=8, n_raw=20, n_show=3)

## 8.5 Quantitative Evaluation: Scoring Metrics on Generated Palettes

The visual gallery and discriminability test show qualitative results. This section
provides the **scientific evaluation**: we compute the same six metrics from the
composite scoring function (Section 3 of the methodology) on model-generated palettes
and compare them to the training labels.

This answers the key question: **Do the generated palettes meet the cartographic quality
criteria defined in the methodology?**

In [ ]:
# ══════════════════════════════════════════════════════════════
# Quantitative Metric Evaluation on Model-Generated Palettes
# ══════════════════════════════════════════════════════════════
#
# For each test sample:
#   1. Generate 20 palette suggestions from the model
#   2. Score each with the 6 cartographic metrics
#   3. Compute best-of-k for k in {1, 3, 5, 10, 20} using GENERATION ORDER
#   4. Apply MMR top-3 and measure diversity-aware quality
#   5. Compare all to ground-truth training label scores
#
# IMPORTANT: best-of-k uses generation order (iid draws), not sorted order.

from research.data_pipeline.score_palettes import compute_composite_score

SCORING_WEIGHTS = {
    'distinguishability': 0.10,
    'basemap_contrast': 0.20,
    'lightness_contrast': 0.30,
    'hue_contrast': 0.30,
    'cvd_robustness': 0.05,
    'perceptual_ordering': 0.05,
}

METRIC_NAMES = [
    'composite_score', 'distinguishability', 'basemap_contrast',
    'lightness_contrast', 'hue_contrast', 'cvd_robustness', 'perceptual_ordering'
]

IDX_TO_SCHEME = {v: k for k, v in SCHEME_TO_IDX.items()}
IDX_TO_NCLASSES = {v: k for k, v in NCLASSES_TO_IDX.items()}
IDX_TO_SCALE = {v: k for k, v in SCALE_TO_IDX.items()}

# == Settings ==
model.eval()
n_suggestions = 20
K_VALUES = [1, 3, 5, 10, 20]

# Storage
all_gen_scores = []
gt_scores = []
best_of_k_results = {k: [] for k in K_VALUES}
mmr_top3_results = []

test_data = test_dataset.data
n_test = len(test_data)
print(f'Evaluating {n_test} test samples with {n_suggestions} suggestions each...')
print(f'Computing best-of-k for k in {K_VALUES}')
print()

basemap_color_cache = {}

for idx in range(n_test):
    if (idx + 1) % 500 == 0 or idx == 0:
        print(f'  [{idx + 1}/{n_test}]')

    row = test_data.iloc[idx]
    patch_id = row['patch_id']
    scheme_type = row['scheme_type']
    n_classes = row['n_classes']

    # Load basemap colors
    if patch_id not in basemap_color_cache:
        color_path = os.path.join(basemap_colors_dir, f'{patch_id}_colors.json')
        if not os.path.exists(color_path):
            continue
        with open(color_path) as f:
            basemap_color_cache[patch_id] = np.array(json.load(f)['colors_lab'])
    basemap_colors_lab = basemap_color_cache[patch_id]

    # Ground-truth score
    gt_lab = np.array(json.loads(row['palette_lab']))
    gt_score_dict = compute_composite_score(gt_lab, basemap_colors_lab, scheme_type, SCORING_WEIGHTS)
    gt_scores.append({
        'patch_id': patch_id, 'scheme_type': scheme_type, 'n_classes': n_classes,
        **gt_score_dict,
    })

    # Generate palettes
    img_tensor, metadata, _, n_col_tensor = test_dataset[idx]
    img_batch = img_tensor.unsqueeze(0).to(device)
    meta_batch = metadata.unsqueeze(0).to(device)

    with torch.no_grad():
        palettes_norm = model.generate(img_batch, meta_batch, n_samples=n_suggestions)
    palettes_norm = palettes_norm.cpu().numpy()

    # Score all suggestions -- keep in GENERATION ORDER for best-of-k
    composites_in_order = []
    scored_for_mmr = []
    for s in range(n_suggestions):
        pal_lab = denormalize_palette(palettes_norm[s], n_classes)
        pal_lab[:, 0] = np.clip(pal_lab[:, 0], 0, 100)
        pal_lab[:, 1] = np.clip(pal_lab[:, 1], -128, 127)
        pal_lab[:, 2] = np.clip(pal_lab[:, 2], -128, 127)

        gen_score_dict = compute_composite_score(pal_lab, basemap_colors_lab, scheme_type, SCORING_WEIGHTS)
        composites_in_order.append(gen_score_dict['composite_score'])
        scored_for_mmr.append((pal_lab.copy(), gen_score_dict['composite_score'], gen_score_dict))

        all_gen_scores.append({
            'patch_id': patch_id, 'scheme_type': scheme_type, 'n_classes': n_classes,
            'suggestion': s, **gen_score_dict,
        })

    # Best-of-k: max of FIRST k samples in generation order
    for k in K_VALUES:
        best_of_k_results[k].append(max(composites_in_order[:k]))

    # MMR top-3
    mmr_selected = mmr_top_k(scored_for_mmr, k=3, lambda_param=0.7)
    mmr_top3_results.append(np.mean([s[1] for s in mmr_selected]))

print(f'\nScored {len(all_gen_scores)} generated palettes and {len(gt_scores)} ground-truth palettes.')

# == Results ==
gen_df = pd.DataFrame(all_gen_scores)
gt_df = pd.DataFrame(gt_scores)

print('\n' + '=' * 80)
print('QUANTITATIVE EVALUATION RESULTS (v4.1)')
print('=' * 80)

# Overall mean-of-all
print('\n--- Overall (all 20 suggestions per sample) ---')
print(f'{"Metric":<25} {"Generated (mean+/-std)":>22} {"Ground Truth (mean+/-std)":>25} {"Delta":>8}')
print('-' * 82)
for m in METRIC_NAMES:
    g_mean = gen_df[m].mean()
    g_std = gen_df[m].std()
    t_mean = gt_df[m].mean()
    t_std = gt_df[m].std()
    delta = g_mean - t_mean
    print(f'{m:<25} {g_mean:>8.3f} +/- {g_std:<8.3f}   {t_mean:>8.3f} +/- {t_std:<8.3f}   {delta:>+.3f}')

# Best-of-k table (correctly computed from generation order)
print('\n--- Best-of-k Composite Score (iid generation order) ---')
print(f'{"k":<6} {"Mean Best Composite":>20} {"P(>=0.70)":>12}')
print('-' * 40)
for k in K_VALUES:
    vals = np.array(best_of_k_results[k])
    print(f'{k:<6} {vals.mean():>20.4f} {(vals >= 0.70).mean():>12.2%}')

# MMR top-3
mmr_arr = np.array(mmr_top3_results)
print(f'\n--- MMR Diversity-Aware Top-3 ---')
print(f'  Mean MMR-top-3 composite: {mmr_arr.mean():.4f} +/- {mmr_arr.std():.4f}')
print(f'  (This is what the final tool would show the user)')

# Per scheme type
print('\n--- By Scheme Type ---')
for scheme in ['sequential', 'diverging']:
    g_sub = gen_df[gen_df['scheme_type'] == scheme]
    t_sub = gt_df[gt_df['scheme_type'] == scheme]
    if len(g_sub) == 0:
        continue
    print(f'\n  {scheme.upper()} (n_gen={len(g_sub)}, n_gt={len(t_sub)})')
    for m in METRIC_NAMES:
        g_mean = g_sub[m].mean()
        t_mean = t_sub[m].mean()
        delta = g_mean - t_mean
        print(f'    {m:<25} gen={g_mean:.3f}  gt={t_mean:.3f}  delta={delta:+.3f}')

# Visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, m in enumerate(METRIC_NAMES):
    ax = axes[i]
    ax.hist(gen_df[m], bins=50, alpha=0.6, label='Generated (all 20)', color='#2196F3', density=True)
    ax.hist(gt_df[m], bins=50, alpha=0.6, label='Ground Truth', color='#4CAF50', density=True)
    ax.set_title(m.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('Score')
    ax.legend(fontsize=8)

# Best-of-k curve in last subplot
ax = axes[7]
k_means = [np.mean(best_of_k_results[k]) for k in K_VALUES]
ax.plot(K_VALUES, k_means, 'bo-', linewidth=2, markersize=8)
gt_mean = gt_df['composite_score'].mean()
ax.axhline(y=gt_mean, color='green', linestyle='--', label=f'GT mean ({gt_mean:.3f})')
ax.set_xlabel('k (number of samples)')
ax.set_ylabel('Mean Best Composite')
ax.set_title('Best-of-k Curve', fontsize=11)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle('CartoPalette v4.1 - Metric Distributions + Best-of-k (Test Split)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['checkpoint_dir'], 'metric_evaluation_v4_1.png'), dpi=150,
            bbox_inches='tight')
plt.show()

# Save results
gen_df.to_csv(os.path.join(CONFIG['checkpoint_dir'], 'generated_palette_scores_v4_1.csv'), index=False)
gt_df.to_csv(os.path.join(CONFIG['checkpoint_dir'], 'groundtruth_palette_scores_v4_1.csv'), index=False)
print(f'\nDetailed results saved to {CONFIG["checkpoint_dir"]}')

## 8.6 Best-of-k Curve Analysis

Systematic analysis of how sample count k affects tool quality.
This justifies the choice of k=20 as default for the CartoPalette tool.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Best-of-k Curve: How many samples does the tool need?
# ══════════════════════════════════════════════════════════════
#
# For k in {1, 3, 5, 10, 20, 50}:
#   - mean(best composite) from FIRST k iid samples
#   - P(best composite >= 0.70)
#   - mean(best distinguishability)
#   - approximate runtime per config
#
# IMPORTANT: best-of-k uses generation order (iid), not sorted.

import time
from research.data_pipeline.score_palettes import compute_composite_score

BOK_WEIGHTS = {
    'distinguishability': 0.10,
    'basemap_contrast': 0.20,
    'lightness_contrast': 0.30,
    'hue_contrast': 0.30,
    'cvd_robustness': 0.05,
    'perceptual_ordering': 0.05,
}

K_SWEEP = [1, 3, 5, 10, 20, 50]
N_EVAL = min(200, len(test_dataset))

model.eval()
indices = np.random.choice(len(test_dataset), N_EVAL, replace=False)

print(f'Generating {max(K_SWEEP)} samples for {N_EVAL} test basemaps...')
all_scores_per_sample = []

t_start = time.time()
for count, idx in enumerate(indices):
    if (count + 1) % 50 == 0:
        print(f'  [{count + 1}/{N_EVAL}]')

    img_tensor, metadata, palette_flat, n_colors = test_dataset[idx]
    n_col = n_colors.item()
    row = test_dataset.data.iloc[idx]
    patch_id = row['patch_id']
    scheme_type = row['scheme_type']

    color_path = os.path.join(basemap_colors_dir, f'{patch_id}_colors.json')
    if not os.path.exists(color_path):
        continue
    with open(color_path) as f:
        bm_colors = np.array(json.load(f)['colors_lab'])

    img_batch = img_tensor.unsqueeze(0).to(device)
    meta_batch = metadata.unsqueeze(0).to(device)

    with torch.no_grad():
        generated = model.generate(img_batch, meta_batch, n_samples=max(K_SWEEP))
    generated = generated.cpu().numpy()

    # Score in GENERATION ORDER (do NOT sort!)
    composites = []
    dists = []
    for s in range(max(K_SWEEP)):
        pal_lab = denormalize_palette(generated[s], n_col)
        pal_lab[:, 0] = np.clip(pal_lab[:, 0], 0, 100)
        pal_lab[:, 1] = np.clip(pal_lab[:, 1], -128, 127)
        pal_lab[:, 2] = np.clip(pal_lab[:, 2], -128, 127)
        sc = compute_composite_score(pal_lab, bm_colors, scheme_type, BOK_WEIGHTS)
        composites.append(sc['composite_score'])
        dists.append(sc['distinguishability'])

    all_scores_per_sample.append({
        'composites': composites,
        'dists': dists,
    })

t_total = time.time() - t_start
t_per_sample = t_total / max(len(all_scores_per_sample), 1)

print(f'\nDone! {len(all_scores_per_sample)} samples evaluated in {t_total:.1f}s')
print(f'Time per config (k={max(K_SWEEP)}): {t_per_sample:.3f}s')

# == Compute best-of-k metrics (from iid generation order) ==
results = []
for k in K_SWEEP:
    # best-of-k = max of FIRST k composites (in generation order)
    best_composites = [max(s['composites'][:k]) for s in all_scores_per_sample]
    # For distinguishability: take dist of the best-composite sample within first k
    best_dists = []
    for s in all_scores_per_sample:
        first_k_composites = s['composites'][:k]
        best_idx_in_k = first_k_composites.index(max(first_k_composites))
        best_dists.append(s['dists'][best_idx_in_k])

    bc = np.array(best_composites)
    bd = np.array(best_dists)
    results.append({
        'k': k,
        'mean_composite': bc.mean(),
        'std_composite': bc.std(),
        'p_above_070': (bc >= 0.70).mean(),
        'p_above_060': (bc >= 0.60).mean(),
        'mean_dist': bd.mean(),
        'est_runtime_ms': t_per_sample * 1000 * k / max(K_SWEEP),
    })

# == Print table ==
print('\n' + '=' * 85)
print('BEST-OF-k CURVE ANALYSIS (iid generation order)')
print('=' * 85)
print(f'{"k":>4} {"Mean Composite":>16} {"Std":>8} {"P(>=0.70)":>10} {"P(>=0.60)":>10} {"Mean Dist":>10} {"~ms/config":>12}')
print('-' * 85)
for r in results:
    print(f'{r["k"]:>4} {r["mean_composite"]:>16.4f} {r["std_composite"]:>8.4f} '
          f'{r["p_above_070"]:>10.2%} {r["p_above_060"]:>10.2%} '
          f'{r["mean_dist"]:>10.4f} {r["est_runtime_ms"]:>12.1f}')

# == Visualize ==
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ks = [r['k'] for r in results]

# Plot 1: Mean best composite
ax = axes[0]
means = [r['mean_composite'] for r in results]
stds = [r['std_composite'] for r in results]
ax.errorbar(ks, means, yerr=stds, fmt='bo-', linewidth=2, markersize=8, capsize=5)
ax.set_xlabel('k (number of samples)', fontsize=12)
ax.set_ylabel('Mean Best Composite Score', fontsize=12)
ax.set_title('Best-of-k Composite Score', fontsize=13)
ax.set_xscale('log')
ax.set_xticks(ks)
ax.set_xticklabels(ks)
ax.grid(True, alpha=0.3)

# Plot 2: P(best >= threshold)
ax = axes[1]
ax.plot(ks, [r['p_above_070'] for r in results], 'rs-', linewidth=2, markersize=8, label='P(>=0.70)')
ax.plot(ks, [r['p_above_060'] for r in results], 'g^-', linewidth=2, markersize=8, label='P(>=0.60)')
ax.set_xlabel('k (number of samples)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('P(best composite >= threshold)', fontsize=13)
ax.set_xscale('log')
ax.set_xticks(ks)
ax.set_xticklabels(ks)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 3: Quality vs Runtime
ax = axes[2]
ax.plot([r['est_runtime_ms'] for r in results], means, 'ko-', linewidth=2, markersize=8)
for r in results:
    ax.annotate(f'k={r["k"]}', (r['est_runtime_ms'], r['mean_composite']),
                textcoords="offset points", xytext=(10, 5), fontsize=10)
ax.set_xlabel('Estimated Runtime (ms/config)', fontsize=12)
ax.set_ylabel('Mean Best Composite Score', fontsize=12)
ax.set_title('Quality vs Runtime Trade-off', fontsize=13)
ax.grid(True, alpha=0.3)

plt.suptitle('CartoPalette v4.1 - Best-of-k Analysis (justifies k=20 default)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['checkpoint_dir'], 'best_of_k_curve_v4_1.png'), dpi=150,
            bbox_inches='tight')
plt.show()

# == Recommendation ==
print('\n--- Recommendation ---')
k20 = next(r for r in results if r['k'] == 20)
k50 = next(r for r in results if r['k'] == 50)
k5 = next(r for r in results if r['k'] == 5)
gain_5_to_20 = k20['mean_composite'] - k5['mean_composite']
gain_20_to_50 = k50['mean_composite'] - k20['mean_composite']
print(f'  k=5  -> k=20: +{gain_5_to_20:.4f} composite (significant gain)')
print(f'  k=20 -> k=50: +{gain_20_to_50:.4f} composite (diminishing returns)')
print(f'  Recommended default: k=20 (best quality/runtime trade-off)')

## 9. Export Final Model

In [ ]:
# Save inference-ready model (without optimizer state)
# Use the best generative model if available, otherwise best recon model
gen_path = os.path.join(CONFIG['checkpoint_dir'], 'best_model_generative.pt')
recon_path = os.path.join(CONFIG['checkpoint_dir'], 'best_model.pt')

if os.path.exists(gen_path):
    source_ckpt = torch.load(gen_path, map_location=device)
    model.load_state_dict(source_ckpt['model_state_dict'])
    source_label = 'GENERATIVE'
    source_epoch = source_ckpt['epoch']
    source_tool_metric = source_ckpt.get('gen_val', {}).get('tool_metric',
                       source_ckpt.get('gen_val', {}).get('composite_score', 0.0))
    print(f'Exporting best GENERATIVE model (epoch {source_epoch}, tool_metric(best-of-20) {source_tool_metric:.4f})')
elif os.path.exists(recon_path):
    source_ckpt = torch.load(recon_path, map_location=device)
    model.load_state_dict(source_ckpt['model_state_dict'])
    source_label = 'RECON'
    source_epoch = source_ckpt['epoch']
    source_val_loss = source_ckpt.get('val_loss', 0.0)
    print(f'Exporting best RECON model (epoch {source_epoch}, val_loss {source_val_loss:.4f})')
else:
    raise FileNotFoundError('No best model found! Run training first.')

export_path = os.path.join(CONFIG['checkpoint_dir'], 'cartopalette_v4.pt')

torch.save({
    'model_state_dict': model.state_dict(),
    'config': CONFIG,
    'source': source_label,
    'source_epoch': source_epoch,
    'encoding_maps': {
        'scheme_to_idx': SCHEME_TO_IDX,
        'scale_to_idx': SCALE_TO_IDX,
        'nclasses_to_idx': NCLASSES_TO_IDX,
    },
    'normalization': {
        'imagenet_mean': IMAGENET_MEAN,
        'imagenet_std': IMAGENET_STD,
        'palette_L_scale': 100.0,
        'palette_ab_scale': 128.0,
    },
}, export_path)

size_mb = os.path.getsize(export_path) / 1e6
print(f'Model exported to: {export_path}')
print(f'Size: {size_mb:.1f} MB')
print(f'\nThis file contains everything needed for inference.')
print(f'Copy it to: v4_CartoPalette/cartopalette/pretrained/cartopalette_v4.pt')

## 10. Quick Test: Generate Palettes for Custom Basemap

In [ ]:
def suggest_palette(model, image_path, scheme_type='sequential', n_classes=5,                    scale_class='regional', n_raw=20, n_show=3):    """Generate palette suggestions using sample+rerank+MMR.        Instead of showing n raw samples, generates n_raw samples,    scores all with composite score, and selects top n_show via MMR    for diverse, high-quality suggestions.    """    from research.data_pipeline.score_palettes import compute_composite_score    WEIGHTS = {        'distinguishability': 0.10,        'basemap_contrast': 0.20,        'lightness_contrast': 0.30,        'hue_contrast': 0.30,        'cvd_robustness': 0.05,        'perceptual_ordering': 0.05,    }    model.eval()    # Load and transform image    image = Image.open(image_path).convert('RGB')    img_tensor = val_transform(image).unsqueeze(0).to(device)    # Build metadata    n_classes_oh = np.zeros(5, dtype=np.float32)    n_classes_oh[NCLASSES_TO_IDX[n_classes]] = 1.0    scheme_oh = np.zeros(2, dtype=np.float32)    scheme_oh[SCHEME_TO_IDX[scheme_type]] = 1.0    scale_oh = np.zeros(3, dtype=np.float32)    scale_oh[SCALE_TO_IDX[scale_class]] = 1.0    metadata = torch.tensor(np.concatenate([n_classes_oh, scheme_oh, scale_oh])).unsqueeze(0).to(device)    # Generate n_raw samples    palettes = model.generate(img_tensor, metadata, n_samples=n_raw)    palettes = palettes.cpu().numpy()    # Load basemap colors for scoring    # Extract dominant colors from the image directly    from skimage import color as skcolor    img_np = np.array(image.resize((64, 64))) / 255.0    img_lab = skcolor.rgb2lab(img_np).reshape(-1, 3)    # Simple k-means for dominant colors    from sklearn.cluster import MiniBatchKMeans    kmeans = MiniBatchKMeans(n_clusters=10, random_state=42, n_init=3)    kmeans.fit(img_lab)    bm_colors = kmeans.cluster_centers_    # Score all samples    scored = []    for s in range(n_raw):        pal_lab = denormalize_palette(palettes[s], n_classes)        sc = compute_composite_score(pal_lab, bm_colors, scheme_type, WEIGHTS)        scored.append((pal_lab, sc['composite_score'], sc))    # MMR top-k selection    mmr_selected = mmr_top_k(scored, k=n_show, lambda_param=0.7)    # Visualize    fig, axes = plt.subplots(1, 1 + n_show, figsize=(3.5 * (1 + n_show), 3.5))    axes[0].imshow(image)    axes[0].set_title('Basemap', fontsize=11)    axes[0].axis('off')    for rank, (pal_lab, s_score, s_detail) in enumerate(mmr_selected):        pal_rgb = lab_to_rgb_simple(pal_lab)        palette_img = np.ones((50, n_classes * 50, 3))        for i in range(n_classes):            palette_img[:, i*50:(i+1)*50] = pal_rgb[i]        axes[rank+1].imshow(palette_img)        dist = s_detail.get('distinguishability', 0)        hex_colors = ['#' + ''.join(f'{int(c*255):02x}' for c in rgb) for rgb in pal_rgb]        axes[rank+1].set_title(f'#{rank+1} (score={s_score:.2f}, d={dist:.2f})', fontsize=9)        axes[rank+1].axis('off')    plt.suptitle(f'CartoPalette v4.1 - {scheme_type}, {n_classes}c, {scale_class}\n'                 f'(sample {n_raw}, rerank+MMR top-{n_show})',                 fontsize=12)    plt.tight_layout()    plt.show()    return mmr_selected# Example usage with a test basemap:test_basemap = os.path.join(basemaps_dir, os.listdir(basemaps_dir)[0])print(f'Testing with: {test_basemap}')# Sequential, 5 classespals_seq = suggest_palette(model, test_basemap, scheme_type='sequential', n_classes=5)# Diverging, 7 classespals_div = suggest_palette(model, test_basemap, scheme_type='diverging', n_classes=7)

## 11. Basemap Discriminability Test

**Critical v4 test:** Do different basemaps produce different palettes?
This was the main failure of v3.

In [ ]:
def basemap_discriminability_test(model, basemaps_dir, n_basemaps=6, n_raw=20, n_show=3):    """Test if the model generates different palettes for different basemaps.        v4.1: Uses sample+rerank+MMR instead of raw samples.    Per basemap: generate n_raw samples, score, MMR top n_show.    """    from research.data_pipeline.score_palettes import compute_composite_score    from skimage import color as skcolor    from sklearn.cluster import MiniBatchKMeans    WEIGHTS = {        'distinguishability': 0.10,        'basemap_contrast': 0.20,        'lightness_contrast': 0.30,        'hue_contrast': 0.30,        'cvd_robustness': 0.05,        'perceptual_ordering': 0.05,    }    model.eval()    # Pick diverse basemaps    all_basemaps = sorted([f for f in os.listdir(basemaps_dir) if f.endswith('.png')])    step = max(1, len(all_basemaps) // n_basemaps)    selected = [all_basemaps[i * step] for i in range(min(n_basemaps, len(all_basemaps)))]    fig, axes = plt.subplots(len(selected), 1 + n_show,                             figsize=(4 * (1 + n_show), 3 * len(selected)))    all_best_palettes = []  # best-of-20 top-1 per basemap for comparison    for row, bm_name in enumerate(selected):        bm_path = os.path.join(basemaps_dir, bm_name)        image = Image.open(bm_path).convert('RGB')        img_tensor = val_transform(image).unsqueeze(0).to(device)        # Metadata (sequential, 5 classes, regional)        n_classes_oh = np.zeros(5, dtype=np.float32)        n_classes_oh[NCLASSES_TO_IDX[5]] = 1.0        scheme_oh = np.zeros(2, dtype=np.float32)        scheme_oh[SCHEME_TO_IDX['sequential']] = 1.0        scale_oh = np.zeros(3, dtype=np.float32)        scale_oh[SCALE_TO_IDX['regional']] = 1.0        metadata = torch.tensor(np.concatenate([n_classes_oh, scheme_oh, scale_oh])).unsqueeze(0).to(device)        # Extract basemap dominant colors        img_np = np.array(image.resize((64, 64))) / 255.0        img_lab = skcolor.rgb2lab(img_np).reshape(-1, 3)        kmeans = MiniBatchKMeans(n_clusters=10, random_state=42, n_init=3)        kmeans.fit(img_lab)        bm_colors = kmeans.cluster_centers_        # Generate n_raw samples        palettes = model.generate(img_tensor, metadata, n_samples=n_raw)        palettes = palettes.cpu().numpy()        # Score all        scored = []        for s in range(n_raw):            pal_lab = denormalize_palette(palettes[s], 5)            sc = compute_composite_score(pal_lab, bm_colors, 'sequential', WEIGHTS)            scored.append((pal_lab, sc['composite_score'], sc))        # MMR top-k        mmr_selected = mmr_top_k(scored, k=n_show, lambda_param=0.7)        all_best_palettes.append(mmr_selected[0][0])  # Store MMR top-1 for comparison        # Show basemap        axes[row, 0].imshow(image)        axes[row, 0].set_title(bm_name[:20], fontsize=9)        axes[row, 0].axis('off')        # Show MMR top-k        for rank, (pal_lab, s_score, s_detail) in enumerate(mmr_selected):            pal_rgb = lab_to_rgb_simple(pal_lab)            palette_img = np.ones((50, 5 * 50, 3))            for i in range(5):                palette_img[:, i*50:(i+1)*50] = pal_rgb[i]            axes[row, rank+1].imshow(palette_img)            axes[row, rank+1].set_title(f'MMR#{rank+1} ({s_score:.2f})', fontsize=9)            axes[row, rank+1].axis('off')    plt.suptitle(f'Basemap Discriminability Test (v4.1)\n'                 f'{n_raw} samples per basemap, MMR top-{n_show}\n'                 f'Each row should show DIFFERENT palettes adapted to that basemap',                 fontsize=13, y=1.03)    plt.tight_layout()    plt.savefig(os.path.join(CONFIG['checkpoint_dir'], 'discriminability_test_v4_1.png'),                dpi=150, bbox_inches='tight')    plt.show()    # Quantitative comparison using MMR top-1 palettes    print('\n--- Quantitative Discriminability (MMR top-1 per basemap) ---')    n = len(all_best_palettes)    distances = []    for i in range(n):        for j in range(i+1, n):            dist = palette_distance(all_best_palettes[i], all_best_palettes[j])            distances.append(dist)    mean_dist = np.mean(distances)    std_dist = np.std(distances)    min_dist = np.min(distances)    max_dist = np.max(distances)    print(f'Mean pairwise palette distance: {mean_dist:.1f} dE (CIELAB)')    print(f'Std: {std_dist:.1f}, Min: {min_dist:.1f}, Max: {max_dist:.1f}')    print()    if mean_dist > 15:        print('PASS: Palettes are meaningfully different across basemaps!')    elif mean_dist > 8:        print('PARTIAL: Some differentiation, but could be stronger.')    else:        print('FAIL: Palettes are too similar - model is not basemap-discriminative.')basemap_discriminability_test(model, basemaps_dir)